# Lesson 1 — Your first look at real data

**Goal:** learn the 10 pandas commands you will use in every data science project, using real air-quality data from 6 Indian cities.

**Time:** about 1.5–2 hours. Don't rush. Run each cell with **Shift + Enter**, read the output, then read the explanation.

**Before you start:** you ran `python -m src.fetch_data`, so `data/processed/daily.csv` exists.

> 💡 Rule for every lesson: after running a cell, *change one thing* (a city, a number, a column) and run it again. That's how it sticks.

## 1. Import libraries

- **pandas** = Excel for Python. It holds tables called *DataFrames*.
- **matplotlib** = draws charts.

`as pd` is just a nickname, so we type `pd` instead of `pandas`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 4)   # default chart size

## 2. Load the data

`read_csv` reads a CSV file into a DataFrame. `parse_dates` tells pandas that the `date` column holds dates, not plain text.

In [ ]:
df = pd.read_csv("../data/processed/daily.csv", parse_dates=["date"])
df.head()      # first 5 rows

## 3. The "first 5 questions" you ask about ANY dataset

| Command | Answers |
|---|---|
| `df.shape` | How many rows and columns? |
| `df.columns` | What are the columns called? |
| `df.info()` | What type is each column? Any missing values? |
| `df.describe()` | Min, max, average of each number column |
| `df.isna().sum()` | How many missing values per column? |

In [ ]:
print("Rows, columns:", df.shape)
df.info()

In [ ]:
df.describe().round(1)

In [ ]:
df.isna().sum()

**Read the output.** Each row = one city on one day. `pm2_5` and `pm10` are tiny particles in the air (µg/m³). `aqi` is the Indian Air Quality Index we calculated in `src/aqi.py` (0 = clean, 500 = severe).

✍️ **Write down:** which column has the biggest maximum? Does any column have missing values?

## 4. Selecting and filtering

- `df["aqi"]` → one column
- `df[df["city"] == "belagavi"]` → only rows where the condition is True

The part inside `df[ ... ]` makes a True/False list, and pandas keeps the True rows.

In [ ]:
belagavi = df[df["city"] == "belagavi"]
print(len(belagavi), "days of data for Belagavi")
belagavi[["date", "pm2_5", "pm10", "aqi", "aqi_category"]].tail()

In [ ]:
# Two conditions: use & (and) or | (or), and wrap each in brackets
bad_days_delhi = df[(df["city"] == "delhi") & (df["aqi"] > 300)]
print(len(bad_days_delhi), "days in Delhi with AQI above 300 ('Very Poor' or worse)")

## 5. Your first chart: AQI over time

In [ ]:
plt.plot(belagavi["date"], belagavi["aqi"])
plt.title("Belagavi — daily AQI")
plt.ylabel("AQI")
plt.show()

✍️ **Look closely:** do you see a pattern that repeats every year? When is the air cleanest? (Hint: think about the monsoon.)

## 6. `groupby` — the most important pandas command

`groupby` = "split the table into groups, calculate something for each group, combine the results".

`df.groupby("city")["aqi"].mean()` → the average AQI for each city.

In [ ]:
city_avg = df.groupby("city")["aqi"].mean().sort_values()
city_avg.round(1)

In [ ]:
city_avg.plot(kind="barh", title="Average AQI by city (Aug 2022 – today)")
plt.xlabel("Average AQI")
plt.show()

## 7. Seasonality: does the month matter?

`.dt.month` pulls the month number (1–12) out of a date. This is our first **feature**: a new column made from existing data. Models learn from features.

In [ ]:
df["month"] = df["date"].dt.month

monthly = df.groupby(["month", "city"])["aqi"].mean().unstack()   # unstack: cities become columns
monthly.plot(marker="o", title="Average AQI by month")
plt.xticks(range(1, 13), ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
plt.ylabel("Average AQI")
plt.show()

This chart is the reason forecasting is possible: **AQI is not random, it follows patterns.** The model you build later will learn these patterns.

## 8. 🕵️ Detective task: is the data trustworthy?

Real data is messy. A good data scientist **checks before trusting**.

Here's a clue: Delhi in winter is one of the most polluted places on Earth. So what should we think about a winter day in Delhi with AQI near 0?

In [ ]:
suspicious = df[(df["city"] == "delhi") & (df["month"].isin([11, 12, 1])) & (df["aqi"] < 20)]
suspicious[["date", "pm2_5", "pm10", "aqi"]]

In [ ]:
# Were ALL cities near-zero on the same days? That would point to a data-source glitch, not real air.
very_low = df[df["aqi"] < 5]
very_low.groupby("date")["city"].count().sort_values(ascending=False).head(5)

✍️ **Your conclusion (write it in a cell below):**
1. Which dates look like a data glitch, and why?
2. Are monsoon days with low AQI also glitches, or real? (Rain washes particles out of the air.)
3. What should we do with glitch days before training a model: delete them, fill them, or flag them?

There's no single right answer. We'll decide together in Lesson 2 (data cleaning). Writing your reasoning down is exactly what the "messy lab data" question in your co-founder form was testing.

## 9. Counting categories

In [ ]:
pd.crosstab(df["city"], df["aqi_category"], normalize="index").mul(100).round(1)

This shows the **% of days** in each AQI category per city. `crosstab` counts combinations of two columns.

## ✅ Exercises (do these yourself — no copying!)

1. Plot the AQI over time for **Kolhapur** and **Delhi** on the same chart. (Hint: call `plt.plot` twice, add `label=` and `plt.legend()`.)
2. Which **single day** had the highest AQI in Mumbai? (Hint: `sort_values("aqi", ascending=False).head(1)`.)
3. Create a column `df["weekday"] = df["date"].dt.day_name()`. Is AQI worse on weekdays than at weekends?
4. Is AQI higher on rainy days? Make a column `rainy = df["precipitation"] > 1` and compare average AQI with `groupby`.
5. **Think:** you want to predict *tomorrow's* AQI. Which columns in this table would be useful? Which might be "cheating"? (Write 3–4 lines.)

## 🎯 You've learned
`read_csv` · `head` · `shape` · `info` · `describe` · `isna` · filtering with conditions · `groupby` · `.dt` date features · `crosstab` · line and bar charts · **checking data quality before trusting it**

**Next lesson:** cleaning the data, and your first "dumb" forecast (the baseline every model must beat).